# Clustering con KMeans en Databricks Free Edition
# Dataset: Adult Income
# Objetivo:
# 1. Preparar datos con Spark
# 2. Construir features para clustering
# 3. Entrenar KMeans
# 4. Evaluar con silhouette
# 5. Registrar parámetros y métricas con MLflow

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, trim, when, avg, count
from pyspark.ml.feature import FeatureHasher
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import gc

In [ ]:
raw_schema = StructType([
    StructField("age", StringType(), True),
    StructField("workclass", StringType(), True),
    StructField("fnlwgt", StringType(), True),
    StructField("education", StringType(), True),
    StructField("education_num", StringType(), True),
    StructField("marital_status", StringType(), True),
    StructField("occupation", StringType(), True),
    StructField("relationship", StringType(), True),
    StructField("race", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("capital_gain", StringType(), True),
    StructField("capital_loss", StringType(), True),
    StructField("hours_per_week", StringType(), True),
    StructField("native_country", StringType(), True),
    StructField("income", StringType(), True)
])

df_raw = (
    spark.read
    .option("header", "false")
    .option("sep", ",")
    .option("ignoreLeadingWhiteSpace", "true")
    .option("ignoreTrailingWhiteSpace", "true")
    .schema(raw_schema)
    .csv("/databricks-datasets/adult/adult.data")
)

for c in df_raw.columns:
    df_raw = df_raw.withColumn(c, trim(col(c)))

display(df_raw.limit(10))
print("Filas originales:", df_raw.count())

In [ ]:
string_cols = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country", "income"
]

for c in string_cols:
    if c != "income":
        df_raw = df_raw.withColumn(c, when(col(c) == "?", None).otherwise(col(c)))

df = (
    df_raw
    .withColumn("age", col("age").cast("int"))
    .withColumn("fnlwgt", col("fnlwgt").cast("int"))
    .withColumn("education_num", col("education_num").cast("int"))
    .withColumn("capital_gain", col("capital_gain").cast("int"))
    .withColumn("capital_loss", col("capital_loss").cast("int"))
    .withColumn("hours_per_week", col("hours_per_week").cast("int"))
    .dropna()
)

print("Filas después de limpieza:", df.count())
display(df.limit(10))

In [ ]:
display(df.groupBy("income").count())
display(df.groupBy("sex").count())
display(df.select("age", "education_num", "hours_per_week", "capital_gain", "capital_loss"))

In [ ]:
income_pd = df.groupBy("income").count().toPandas()

plt.figure(figsize=(6,4))
plt.bar(income_pd["income"], income_pd["count"])
plt.title("Distribución de income")
plt.xlabel("income")
plt.ylabel("count")
plt.show()

In [ ]:
categorical_cols = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country"
]

numeric_cols = [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week"
]

In [ ]:
HASH_NUM_FEATURES = 256

# FeatureHasher no requiere fit — convierte numéricas y categóricas en un vector sparse
feature_hasher = FeatureHasher(
    inputCols=numeric_cols + categorical_cols,
    outputCol="features",
    numFeatures=HASH_NUM_FEATURES
)

In [ ]:
def cleanup_kmeans_vars():
    for name in [
        "kmeans_model", "predictions", "df_features",
        "cluster_sample_pd", "cluster_sizes_pd", "summary_pd"
    ]:
        if name in globals():
            del globals()[name]
    gc.collect()


def train_evaluate_kmeans(k, max_iter=20, seed=42):
    cleanup_kmeans_vars()

    # Preparar features con FeatureHasher (no requiere fit, evita Py4J whitelist)
    df_features = feature_hasher.transform(df)

    kmeans = KMeans(
        featuresCol="features",
        predictionCol="prediction",
        k=k,
        maxIter=max_iter,
        seed=seed
    )

    with mlflow.start_run(run_name=f"kmeans_k_{k}"):
        kmeans_model = kmeans.fit(df_features)
        predictions = kmeans_model.transform(df_features)

        evaluator = ClusteringEvaluator(
            featuresCol="features",
            predictionCol="prediction",
            metricName="silhouette",
            distanceMeasure="squaredEuclidean"
        )

        silhouette = evaluator.evaluate(predictions)

        training_cost = kmeans_model.summary.trainingCost
        cluster_sizes = kmeans_model.summary.clusterSizes
        centers = kmeans_model.clusterCenters()

        mlflow.log_param("algorithm", "KMeans")
        mlflow.log_param("k", k)
        mlflow.log_param("maxIter", max_iter)
        mlflow.log_param("seed", seed)
        mlflow.log_param("hash_num_features", HASH_NUM_FEATURES)

        mlflow.log_metric("silhouette", silhouette)
        mlflow.log_metric("training_cost", float(training_cost))

        result = {
            "k": k,
            "maxIter": max_iter,
            "silhouette": silhouette,
            "training_cost": float(training_cost)
        }

        cluster_sample_pd = (
            predictions
            .select(
                "age",
                "education",
                "occupation",
                "hours_per_week",
                "income",
                "prediction"
            )
            .limit(30)
            .toPandas()
        )

        cluster_sizes_pd = pd.DataFrame({
            "cluster": list(range(len(cluster_sizes))),
            "size": list(cluster_sizes)
        })

        centers_pd = pd.DataFrame(
            [center.toArray().tolist() for center in centers]
        )

        summary_pd = (
            predictions
            .groupBy("prediction", "income")
            .count()
            .orderBy("prediction", "income")
            .toPandas()
        )

        del predictions
        del df_features
        del kmeans_model
        gc.collect()

        return result, cluster_sample_pd, cluster_sizes_pd, centers_pd, summary_pd

In [ ]:
result_k3, sample_k3, sizes_k3, centers_k3, summary_k3 = train_evaluate_kmeans(
    k=3,
    max_iter=20,
    seed=42
)

print(result_k3)
display(sample_k3)
display(sizes_k3)
display(summary_k3)

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(sizes_k3["cluster"].astype(str), sizes_k3["size"])
plt.title("Tamaño de clusters para k=3")
plt.xlabel("Cluster")
plt.ylabel("Cantidad de registros")
plt.show()

In [ ]:
summary_pivot_k3 = summary_k3.pivot(index="prediction", columns="income", values="count").fillna(0)
summary_pivot_k3.plot(kind="bar", figsize=(8,5))
plt.title("Distribución de income por cluster (k=3)")
plt.xlabel("Cluster")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
results = []

for k in [2, 3, 4, 5, 6]:
    result, _, _, _, _ = train_evaluate_kmeans(k=k, max_iter=20, seed=42)
    results.append(result)

results_pd = pd.DataFrame(results)
display(spark.createDataFrame(results_pd))

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(results_pd["k"], results_pd["silhouette"], marker="o")
plt.title("Silhouette por valor de k")
plt.xlabel("k")
plt.ylabel("Silhouette")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(results_pd["k"], results_pd["training_cost"], marker="o")
plt.title("Training cost por valor de k")
plt.xlabel("k")
plt.ylabel("Training cost")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(results_pd["k"], results_pd["training_cost"], marker="o")
plt.title("Training cost por valor de k")
plt.xlabel("k")
plt.ylabel("Training cost")
plt.grid(True)
plt.show()

In [ ]:
# Seleccionar el mejor k por silhouette
best_k_row = results_pd.sort_values("silhouette", ascending=False).iloc[0]
best_k = int(best_k_row["k"])

best_result, best_sample, best_sizes, best_centers, best_summary = train_evaluate_kmeans(
    k=best_k,
    max_iter=20,
    seed=42
)

print(f"Mejor k = {best_k}")
print(best_result)
display(best_sample)
display(best_sizes)
display(best_summary)

In [ ]:
cluster_profile = (
    df.join(
        spark.createDataFrame(best_sample[["age", "education", "occupation", "hours_per_week", "income", "prediction"]]),
        on=["age", "education", "occupation", "hours_per_week", "income"],
        how="inner"
    )
    .groupBy("prediction")
    .agg(
        avg("age").alias("avg_age"),
        avg("education_num").alias("avg_education_num"),
        avg("hours_per_week").alias("avg_hours_per_week"),
        avg("capital_gain").alias("avg_capital_gain")
    )
    .orderBy("prediction")
)

display(cluster_profile)

In [ ]:
# Reentrenar para perfilar clusters completos
df_features = feature_hasher.transform(df)
kmeans = KMeans(featuresCol="features", predictionCol="prediction", k=best_k, maxIter=20, seed=42)
kmeans_model = kmeans.fit(df_features)
predictions_best = kmeans_model.transform(df_features)

cluster_profile = (
    predictions_best
    .groupBy("prediction")
    .agg(
        avg("age").alias("avg_age"),
        avg("education_num").alias("avg_education_num"),
        avg("hours_per_week").alias("avg_hours_per_week"),
        avg("capital_gain").alias("avg_capital_gain"),
        count("*").alias("records")
    )
    .orderBy("prediction")
)

display(cluster_profile)

del predictions_best
del df_features
del kmeans_model
gc.collect()